# Panel de pruebas · FlowDataLab

Compara todo lo que han probado los hilos de experimentos, con **la misma métrica y los mismos folds** (`Data/folds.csv`, CV top-3 por controlador).

- **Futuro** (`futuro/pruebas/resultados.jsonl`): modelos que usan otros snapshots de la sesión (rompen la regla 1 a propósito). Referencia: `lgb_future`, CV 0.9010.
- **Solo snapshot** (`pruebas/resultados.jsonl`): la línea de `main`. Referencia: `lgb_adri`, CV 0.6569.

Cada fila es una ejecución registrada con `registrar()`; si un nombre se repite, vale la última.
Columnas: `cv` (media de snapshots), `min_atco` (peor controlador, lo que más se parece a jugarse el privado con un controlador nuevo), `std_folds` (estabilidad).
Ojo: una prueba con 1 semilla se compara con la `*_ref` de su hilo (también 1 semilla), no con la referencia de 2–3 semillas.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "Data").exists():
    ROOT = ROOT.parent
FUT_P, SNAP_P = ROOT / "futuro" / "pruebas", ROOT / "pruebas"
sys.path[:0] = [str(ROOT / "cv"), str(ROOT / "stacking"), str(SNAP_P), str(FUT_P)]
pd.set_option("display.width", 200, "display.max_colwidth", 70)


def cargar(carpeta):
    filas = [json.loads(l) for l in open(carpeta / "resultados.jsonl")]
    r = pd.DataFrame(filas).drop_duplicates("nombre", keep="last").reset_index(drop=True)
    return r


def tabla(r, ref):
    t = r[["hilo", "nombre", "cv", "min_atco", "std_folds", "n_features", "tiene_preds", "descripcion"]].copy()
    t.insert(3, "vs_ref", (t.cv - ref).round(4))
    return t.sort_values("cv", ascending=False).reset_index(drop=True)


fut, snap = cargar(FUT_P), cargar(SNAP_P)
print(f"futuro: {len(fut)} ejecuciones · solo snapshot: {len(snap)} ejecuciones")

## 1 · Futuro: todas las pruebas

In [ ]:
REF_FUT = fut.set_index("nombre").cv.get("lgb_future", np.nan)
tabla(fut, REF_FUT).style.format({"cv": "{:.4f}", "vs_ref": "{:+.4f}", "min_atco": "{:.3f}", "std_folds": "{:.3f}"}) \
    .background_gradient(subset=["cv", "min_atco"], cmap="Greens")

### Comparables con `lgb_future` (3 semillas: 0, 1 y 2)

Solo las ejecuciones con las mismas semillas que `lgb_future`: las `*_s3` (repetidas con `futuro/pruebas/s3.py`, que reaprovecha la semilla 0 ya entrenada), las `*_best` de los hilos y los stacks. Esta es la tabla para decidir.

In [ ]:
comp = fut[(fut.hilo == "referencia") | fut.nombre.str.endswith(("_s3", "_best")) | fut.nombre.str.startswith("stackf")]
tabla(comp, REF_FUT).style.format({"cv": "{:.4f}", "vs_ref": "{:+.4f}", "min_atco": "{:.3f}", "std_folds": "{:.3f}"}) \
    .background_gradient(subset=["cv", "min_atco"], cmap="Greens")

### Mejora frente a la referencia de cada hilo (1 semilla contra 1 semilla)

In [ ]:
def vs_ref_hilo(r):
    out = []
    for hilo, g in r[r.hilo != "referencia"].groupby("hilo"):
        refs = g[g.nombre.str.endswith("_ref")]
        if refs.empty:
            continue
        ref = refs.iloc[-1]
        for _, x in g[~g.nombre.str.endswith("_ref")].iterrows():
            out.append(dict(hilo=hilo, nombre=x.nombre, d_cv=x.cv - ref.cv, d_min_atco=x.min_atco - ref.min_atco))
    return pd.DataFrame(out).sort_values("d_cv")


def grafico_vs_ref(r, titulo):
    v = vs_ref_hilo(r)
    if v.empty:
        print("todavía no hay pruebas con referencia de su hilo"); return
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(v) + 1))
    ax.barh(v.nombre, v.d_cv, color=np.where(v.d_cv > 0, "#2e7d32", "#b0b7c3"))
    ax.scatter(v.d_min_atco, v.nombre, color="#c62828", s=18, zorder=3, label="Δ peor controlador")
    ax.axvline(0, color="#555", lw=1); ax.set_xlabel("Δ frente a la *_ref de su hilo"); ax.set_title(titulo)
    ax.legend(loc="lower right", fontsize=8); ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()


grafico_vs_ref(fut, "Futuro: Δ CV (barras) y Δ peor controlador (puntos)")

## 2 · Futuro: acierto por tipo de la tarea correcta

Dónde gana o pierde cada prueba. Los tipos van ordenados por frecuencia; las coordinaciones son las que quedan por arreglar.

In [ ]:
ORDEN = ["transfer_aircraft", "assume_aircraft", "zone_conflict", "entry_coordination", "return_to_route",
         "exit_coordination", "exit_conditions", "conflict_resolution", "qos_improvement", "exit_conflict_resolution",
         "non_conformance_resolution", "entry_conflict_resolution"]


def por_tipo(r, top=12):
    r = r.sort_values("cv", ascending=False).head(top)
    t = pd.DataFrame(list(r.por_tipo), index=r.nombre)
    return t[[c for c in ORDEN if c in t.columns]]


por_tipo(fut).style.format("{:.2f}").background_gradient(cmap="RdYlGn", axis=None, vmin=0.4, vmax=1)

## 3 · Futuro: acierto por controlador

El privado es un controlador nuevo (023): una prueba que sube la media pero hunde a algún controlador es arriesgada.

In [ ]:
def por_atco(r, top=12):
    r = r.sort_values("cv", ascending=False).head(top)
    t = pd.DataFrame(list(r.por_atco), index=r.nombre)
    return t[sorted(t.columns, key=int)]


por_atco(fut).style.format("{:.2f}").background_gradient(cmap="RdYlGn", axis=None, vmin=0.6, vmax=1)

## 4 · Futuro: stacking con lo nuevo

Mismo `stacking/stack.py` que en `main` (media ponderada con CV anidada y meta-ranker), sobre los modelos de `futuro/oof` más los de `futuro/pruebas/oof` **que tienen preds de test**. Las salidas van a `futuro/pruebas/` como `stackf_*` y se registran en el panel. Tarda unos minutos; `HACER_STACK = False` para saltarlo.

In [ ]:
HACER_STACK = True
MODELOS = None   # None = todos los que tienen oof + preds; o una lista, p.ej. ["lgb_future", "nn_future", "f2_best"]

if HACER_STACK:
    import time, stack
    from base_fut import registrar
    stack.DIRS, stack.OUT = [ROOT / "futuro", FUT_P], FUT_P
    guardado = {}
    stack.save_outputs = lambda name, oof, preds, out: guardado.update({name: (oof, preds)})
    t0 = time.time()
    argv = sys.argv; sys.argv = ["stack.py"] + (MODELOS or [])
    try:
        stack.main()
    finally:
        sys.argv = argv
    usados = stack.available(MODELOS or [], False)
    from metric import save_outputs
    for name, (oof, preds) in guardado.items():
        nombre = name.replace("stack", "stackf")
        save_outputs(nombre, oof, preds, out=FUT_P)
        registrar(nombre, hilo="stack", descripcion="stacking de: " + ", ".join(usados), oof_df=oof,
                  segundos=time.time() - t0, n_features=len(usados))
    fut = cargar(FUT_P)

### Correlación de aciertos entre los modelos con preds

Cuanto más baja, más puede aportar un modelo al stacking aunque su CV sea algo menor.

In [ ]:
from metric import hits


def corr_aciertos(carpetas, nombres):
    H = {}
    for n in nombres:
        for c in carpetas:
            p = c / "oof" / f"{n}.csv"
            if p.exists():
                H[n] = hits(pd.read_csv(p)); break
    return pd.DataFrame(H).corr()


con_preds = [n for n in fut.nombre if (FUT_P / "preds" / f"{n}.csv").exists() or (ROOT / "futuro" / "preds" / f"{n}.csv").exists()]
con_preds = [n for n in con_preds if not n.startswith(("futuro_stack", "stackf"))]
corr_aciertos([ROOT / "futuro", FUT_P], con_preds).style.format("{:.2f}").background_gradient(cmap="Blues", vmin=0.5, vmax=1)

## 5 · Solo snapshot (línea de main)

Pruebas de la primera tanda, sin futuro. Referencia `lgb_adri` 0.6569 (2 semillas) = `ref_s2`; las `*_ref` de 1 semilla dan 0.6596.

In [ ]:
REF_SNAP = snap.set_index("nombre").cv.get("lgb_adri", np.nan)
tabla(snap, REF_SNAP).style.format({"cv": "{:.4f}", "vs_ref": "{:+.4f}", "min_atco": "{:.3f}", "std_folds": "{:.3f}"}) \
    .background_gradient(subset=["cv", "min_atco"], cmap="Greens")

In [ ]:
grafico_vs_ref(snap, "Solo snapshot: Δ CV (barras) y Δ peor controlador (puntos)")
por_tipo(snap).style.format("{:.2f}").background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)

## 6 · Notas de cada hilo

In [ ]:
from IPython.display import Markdown, display
for carpeta in (FUT_P, SNAP_P):
    for f in sorted((carpeta / "notas").glob("*.md")):
        display(Markdown(f"### {f.relative_to(ROOT)}\n\n" + f.read_text()))

## 7 · Conclusiones

_(se rellenan al final de la tanda)_